# Explore Scoring Model Behavior

Requirements: `pip install transformers supabase python-dotenv`

You should already ha

In [14]:
import os
import ipywidgets as widgets
from IPython.display import HTML

import torch
import numpy as np
import pandas as pd
from transformers import pipeline
from supabase import create_client, Client
from dotenv import load_dotenv
from tqdm.auto import tqdm

from models import ModernBERT_v2, ModelInput

load_dotenv()
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")
supabase = create_client(supabase_url=url, supabase_key=key)

In [2]:
# Get chunk data
chunk_df = pd.read_csv(
    "/home/jovyan/active-projects/shared-resources/strapi-exports/data/strapi_data.csv",
    index_col=0,
)

# There are some duplicate chunks. Likely multiple pages linked to the same chunk.
chunk_df = chunk_df.loc[~chunk_df.index.duplicated(keep="first"), :]
chunk_df.index.value_counts()

chunk_slug
Discussion-II-5852                                       1
Structure-and-Function-of-the-Respiratory-System-6247    1
The-Conducting-Airways-6248                              1
The-Respiratory-Airways-6249                             1
Gas-Exchange-6250                                        1
                                                        ..
16-Training-Records-2031                                 1
17-Training-Program-Evaluation-2032                      1
31-Job-Training-and-Transfer-Training-2023               1
33-Refresher-Training-2024                               1
41-Requirements-and-Placement-2039                       1
Name: count, Length: 585, dtype: int64

In [23]:
# Get logs data
client_names = ["hinze_rmp_summer_2025", "middle_georgia"]

logs_response = (
    supabase.table("logs")
    .select("*")
    # .in_("client_name", client_names)  # Optionally filter by client_name
    .eq("api_endpoint", "/score/answer")  # CRI Endpoint
    .order("created_at", desc=True)  # Optionally sort by most recent API calls
    .limit(1_000)  # Number of samples to collect
    .execute()
)

logs_df = pd.json_normalize(logs_response.data)

# Join API query data with content data extracted from Strapi (to get volume slug, page slug, chunk text, question, reference answer)
df = logs_df.merge(
    chunk_df, left_on="request_body.chunk_slug", right_on="chunk_slug", how="left"
)

# Construct DataFrame
columns_to_keep = [
    "volume_slug",
    "request_body.page_slug",
    "request_body.chunk_slug",
    "chunk_text",
    "question",
    "answer",
    "request_body.answer",
    "response_body.level",
    "response_body.score",
    "response_body.feedback",
    "response_body.is_passing",
]

df = df[columns_to_keep]
df = df.rename(
    columns={
        "answer": "reference",
        "request_body.answer": "candidate",
        "chunk_text": "text",
    }
)

# Clean up column names
df.columns = [
    col.replace("request_body.", "").replace("response_body.", "") for col in df.columns
]

# Remove rows without necessary information
df = df[~df["question"].isna() & ~df["text"].isna() & ~df["reference"].isna()]  # Some rows have NaN values

df

,volume_slug,page_slug,chunk_slug,text,question,reference,candidate,level,score,feedback,is_passing
0,research-methods-in-psychology-demo,13-drawing-conclusions-and-reporting-the-results,Reporting-the-Results-1588t,The final step in the research process involve...,What are the different ways to report research...,Research findings can be reported through peer...,The different ways to report research findings...,3,2.468085,Good job. You included some relevant details f...,True
1,research-methods-in-psychology-demo,13-drawing-conclusions-and-reporting-the-results,Reporting-the-Results-1588t,The final step in the research process involve...,What are the different ways to report research...,Research findings can be reported through peer...,The different ways to report research findings...,4,3.345560,Excellent job! Your response demonstrates deep...,True
2,research-methods-in-psychology-demo,13-drawing-conclusions-and-reporting-the-results,Reporting-the-Results-1588t,The final step in the research process involve...,What are the different ways to report research...,Research findings can be reported through peer...,The different ways to report research findings...,4,3.367953,Excellent job! Your response demonstrates deep...,True
3,research-methods-in-psychology-demo,12-analyzing-the-data,Descriptive-Statistics-1583t,Descriptive statistics are used to organize or...,What is the purpose of correlation coefficient...,Correlation coefficients describe the strength...,The purpose of correlation coefficients in des...,4,3.549463,Excellent job! Your response demonstrates deep...,True
4,research-methods-in-psychology-demo,11-designing-a-research-study,Non-Experimental-Research-1575t,Researchers who are simply interested in descr...,What can non-experimental research be used for...,Non-experimental research can be used to descr...,Non-experimental research can be used in the s...,4,3.631720,Excellent job! Your response demonstrates deep...,True
...,...,...,...,...,...,...,...,...,...,...,...
995,research-methods-in-psychology,59-additional-considerations,Errors-in-Null-Hypothesis-Testing-5728,"In null hypothesis testing, the researcher tri...",What is a Type I error in null hypothesis test...,"Rejecting the null hypothesis when it is true,...","Rejecting the null hypothesis when it is true,...",3,3.003048,Good job. You included some relevant details f...,True
996,research-methods-in-psychology,59-additional-considerations,Errors-in-Null-Hypothesis-Testing-5728,"In null hypothesis testing, the researcher tri...",What is a Type I error in null hypothesis test...,"Rejecting the null hypothesis when it is true,...",think what you have is correct but it is actua...,2,1.904462,"You are on the right track, but your response ...",False
997,research-methods-in-psychology,1-methods-of-knowing,Rationalism-and-Empiricism-1437t,Rationalism\n\nRationalism involves using logi...,What is the scientific method and how is it us...,The scientific method is a process of systemat...,das,1,1.257368,Your response is missing many relevant details...,False
998,natural-language-processing,the-tool-for-the-automatic-analysis-of-lexical...,Overview-of-findings-4860,Overview of findings\n\nThis study reported on...,What percentage of the variance in lexical pro...,28% of the variance in lexical proficiency sco...,fsd,1,1.293466,Your response is missing many relevant details...,False


## Re-score the data

In [4]:
mb_w_reference_answer_path = "/home/jovyan/active-projects/itell-question-generation/results/modernbert_multirc_reading_engagement"

MBV2 = ModernBERT_v2(mb_w_reference_answer_path)

Device set to use cuda


In [25]:
def score(df, pipe):
    all_preds = []

    for ex in tqdm(df.to_dict("records")):
        model_input = ModelInput.from_dict(ex)
        pred = pipe(model_input)
        all_preds.append(pred)

    return all_preds


df["new_score"] = score(df, MBV2)

  0%|          | 0/983 [00:00<?, ?it/s]

In [44]:
# Feel free to modify filters

PASSING_THRESHOLD = 2.5  # Scores >= this value are passing

filters = (
    (abs(df["score"] - df["new_score"]) > 1.0)  # scores differ by more than 1.0
    | ((df["score"] > PASSING_THRESHOLD) & (df["new_score"] < PASSING_THRESHOLD))  # scores on different sides of threshold
    | ((df["score"] < PASSING_THRESHOLD) & (df["new_score"] > PASSING_THRESHOLD))  # scores on different sides of threshold
) 

# For example, this will select scores that are within TOLERANCE of TARGET_SCORE
TARGET_SCORE = 1.8
TOLERANCE = 0.2
filters = (df["new_score"] - TARGET_SCORE).abs() < TOLERANCE

# Apply filters
df_filtered = df[filters]

df_filtered

,volume_slug,page_slug,chunk_slug,text,question,reference,candidate,level,score,feedback,is_passing,new_score
173,examiner-s-manual-standardization,requirements-for-student-participants,Requirements-for-Student-Participants-6183,The purpose of the CELF-6 Standardization is t...,What is the purpose of the CELF-6 Standardizat...,The purpose of the CELF-6 Standardization is t...,To not have a diagnosis of color blindness,4,3.516423,Excellent job! Your response demonstrates deep...,True,1.791737
187,natural-language-processing,the-tool-for-the-automatic-analysis-of-lexical...,Discussion-4857,This study introduces and helps validate TAALE...,What is the purpose of TAALES 2.0 as described...,TAALES 2.0 is a tool for measuring lexical sop...,It is an easy to use freely available versatil...,2,1.906836,"You are on the right track, but your response ...",False,1.679042
189,natural-language-processing,vances-in-transformation-based-part-of-speech-...,Unknown-Words-5136,"In addition to not being lexicalized, another ...",What approach was used in the transformation-b...,The initial state annotator naively labels the...,A number of suffixes and important features ar...,3,2.899514,Good job. You included some relevant details f...,True,1.880504
214,natural-language-processing,text-coherence-and-judgments-of-essay-quality-...,Discussion-II-5867,Multiple Regression Training Set\n\nA linear r...,What percentage of the variance in human evalu...,80% of the variance.,81% of the variance,2,2.067881,"You are on the right track, but your response ...",False,1.744666
225,natural-language-processing,sentiment-analysis-and-social-cognition-engine...,Results-5769,Movie review corpus\n\nLIWC indices\n\nMANOVA\...,What was the accuracy rate of the DFA using th...,The accuracy rate was 85.0%.,84.2%-85% accuracy,2,2.004777,"You are on the right track, but your response ...",False,1.916633
237,research-methods-in-psychology,5-experimental-and-clinical-psychologists,Clinical-Psychologists-1466t,Psychology is the scientific study of behavior...,What are empirically supported treatments in p...,Treatments that have been shown to work throug...,Treatements that have been shown to work are c...,3,2.771466,Good job. You included some relevant details f...,True,1.978881
272,natural-language-processing,vances-in-transformation-based-part-of-speech-...,Unknown-Words-5136,"In addition to not being lexicalized, another ...",What approach was used in the transformation-b...,The initial state annotator naively labels the...,The transformation-based tagger constructed ru...,4,3.336455,Excellent job! Your response demonstrates deep...,True,1.943286
274,natural-language-processing,vances-in-transformation-based-part-of-speech-...,Unknown-Words-5136,"In addition to not being lexicalized, another ...",What approach was used in the transformation-b...,The initial state annotator naively labels the...,A statistical approach was used in the transfo...,4,3.251928,Excellent job! Your response demonstrates deep...,True,1.771421
280,mars-petcare,mars-petcare-i,Experiment-Summary-5912,We are planning to cook 126* unique combinatio...,What are the criteria for ranking the top comb...,The criteria include visual quality based on r...,top three solutions for visual representations...,2,1.946577,"You are on the right track, but your response ...",False,1.716063
288,natural-language-processing,vances-in-transformation-based-part-of-speech-...,Unknown-Words-5136,"In addition to not being lexicalized, another ...",What approach was used in the transformation-b...,The initial state annotator naively labels the...,It assigns the most likely tag based on the tr...,4,3.240489,Excellent job! Your response demonstrates deep...,True,1.778334


In [45]:
current_idx = 0

def create_display(df, idx):
    """Create the display for a given row index"""
    if idx < 0 or idx >= len(df):
        return None

    row = df.iloc[idx]

    # Round scores to 2 decimal places
    score = round(row["score"], 2)
    new_score = round(row["new_score"], 2)

    # Determine passing status and styling
    score_passing = score >= PASSING_THRESHOLD
    new_score_passing = new_score >= PASSING_THRESHOLD

    score_bg = "#c8e6c9" if score_passing else "#ffcdd2"
    new_score_bg = "#c8e6c9" if new_score_passing else "#ffcdd2"

    html_content = f"""
    <div style="font-family: Arial, sans-serif; line-height: 1.6;">
        <div style="background-color: #f0f0f0; padding: 10px; border-radius: 5px; margin-bottom: 15px;">
            <strong>Row {idx + 1} of {len(df)}</strong>
        </div>
        
        <div style="background-color: #e8f4f8; padding: 12px; border-left: 4px solid #2196F3; margin-bottom: 15px;">
            <strong>Question:</strong><br>
            {row["question"]}
        </div>
        
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;">
            <div style="background-color: {score_bg}; padding: 10px; border-radius: 8px; border: 2px solid #388e3c; text-align: center;">
                <div style="font-size: 12px; color: #333; margin-bottom: 8px;">Original Score</div>
                <div style="font-size: 18px; font-weight: bold; color: #1b5e20;">{score}</div>
            </div>
            
            <div style="background-color: {new_score_bg}; padding: 10px; border-radius: 8px; border: 2px solid #388e3c; text-align: center;">
                <div style="font-size: 12px; color: #333; margin-bottom: 8px;">New Score</div>
                <div style="font-size: 18px; font-weight: bold; color: #1b5e20;">{new_score}</div>
            </div>
        </div>
                
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
            <div style="background-color: #e8f5e9; padding: 15px; border: 2px solid #4CAF50; border-radius: 5px;">
                <h3 style="margin-top: 0; color: #2e7d32;">Reference Answer</h3>
                <div style="font-size: 14px; line-height: 1.8;">
                    {row["reference"]}
                </div>
            </div>
            
            <div style="background-color: #fce4ec; padding: 15px; border: 2px solid #E91E63; border-radius: 5px;">
                <h3 style="margin-top: 0; color: #880e4f;">Student Response</h3>
                <div style="font-size: 14px; line-height: 1.8;">
                    {row["candidate"]}
                </div>
            </div>
        </div>

        <hr style="margin: 20px 0;">

        <div style="background-color: #fff3e0; padding: 12px; border-left: 4px solid #FF9800; margin-bottom: 15px;">
            <strong>Text/Context:</strong><br>
            {row["text"]}
        </div>

    </div>
    """

    return html_content


# Create output widget
output = widgets.Output()

# Create buttons
prev_button = widgets.Button(description="← Previous", button_style="info")
next_button = widgets.Button(description="Next →", button_style="info")
status_label = widgets.Label(value=f"Row 1 of {len(df_filtered)}")


def on_prev_clicked(b):
    global current_idx
    current_idx = max(0, current_idx - 1)
    update_display()


def on_next_clicked(b):
    global current_idx
    current_idx = min(len(df_filtered) - 1, current_idx + 1)
    update_display()


def update_display():
    output.clear_output(wait=True)
    html_content = create_display(df_filtered, current_idx)
    status_label.value = f"Row {current_idx + 1} of {len(df_filtered)}"
    with output:
        display(HTML(html_content))


prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

# Create layout
button_box = widgets.HBox([prev_button, status_label, next_button])
layout = widgets.VBox([button_box, output])

# Initial display
update_display()

# Show the interface
display(layout)

LH - I am thinking the following for thresholds:
1. (-inf, 1.5)
2. [1.5, 2.1)
3. [2.1, 3.3)
4. [3.3, +inf)